# 第一阶段：金融数据探索与理解

Notebook记录学习金融数据的基本处理和分析方法。

**学习目标**：
1. 理解金融数据的结构和特点
2. 掌握收益率、波动率等核心指标的计算
3. 学会使用技术指标进行初步分析

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import StockDataLoader, PortfolioAnalyzer

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("环境加载成功！")

## 1. 获取股票数据

使用yfinance获取股票历史数据。这里以苹果公司(AAPL)为例。

In [ ]:
# 创建数据加载器
loader = StockDataLoader()

# 获取苹果股票数据（2020-2024年）
symbol = 'AAPL'
df = loader.fetch_stock_data(symbol, '2020-01-01', '2024-01-01')

print(f"数据形状: {df.shape}")
print("\n前5行数据:")
df.head()

## 2. 计算收益率

**思考**：为什么要同时计算简单收益率和对数收益率？
- 简单收益率：直观，适合描述单期收益
- 对数收益率：时间可加性，适合多期复合计算

In [ ]:
# 计算收益率
df = loader.calculate_returns(df)

# 可视化收益率分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 简单收益率分布
axes[0].hist(df['Simple_Return'].dropna(), bins=50, alpha=0.7, edgecolor='black')
axes[0].set_title('简单收益率分布')
axes[0].set_xlabel('收益率')
axes[0].set_ylabel('频次')
axes[0].axvline(df['Simple_Return'].mean(), color='r', linestyle='--', label=f'均值: {df["Simple_Return"].mean():.4f}')
axes[0].legend()

# 对数收益率分布
axes[1].hist(df['Log_Return'].dropna(), bins=50, alpha=0.7, edgecolor='black', color='orange')
axes[1].set_title('对数收益率分布')
axes[1].set_xlabel('收益率')
axes[1].set_ylabel('频次')
axes[1].axvline(df['Log_Return'].mean(), color='r', linestyle='--', label=f'均值: {df["Log_Return"].mean():.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n收益率统计:")
print(df[['Simple_Return', 'Log_Return']].describe())

## 3. 计算波动率

**思考**：波动率为什么重要？
- 衡量风险的核心指标
- 高波动率意味着不确定性增加
- 常用于期权定价、风险管理

In [ ]:
# 计算不同窗口的滚动波动率
df = loader.calculate_volatility(df, window=20)
df = loader.calculate_volatility(df, window=60)

# 可视化价格和波动率
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 价格走势
axes[0].plot(df.index, df['Close'], label='收盘价', color='blue')
axes[0].set_title(f'{symbol} 股价走势')
axes[0].set_ylabel('价格 ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 波动率
axes[1].plot(df.index, df['Volatility_20d'], label='20日波动率', alpha=0.8)
axes[1].plot(df.index, df['Volatility_60d'], label='60日波动率', alpha=0.8)
axes[1].set_title('滚动波动率（年化）')
axes[1].set_xlabel('日期')
axes[1].set_ylabel('波动率')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 观察：波动率聚类现象
print("\n观察：注意2020年3月（疫情爆发）和2022年的波动率飙升")

## 4. 移动平均线与趋势分析

**思考**：不同周期的MA如何帮助我们判断趋势？
- MA5/MA10：短期趋势
- MA20：中期趋势
- MA60：长期趋势
- 金叉/死叉：MA交叉产生的交易信号

In [ ]:
# 计算移动平均线
df = loader.calculate_moving_averages(df, windows=[5, 20, 60])

# 可视化
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label='收盘价', alpha=0.8, linewidth=1)
plt.plot(df.index, df['MA_5'], label='MA5', alpha=0.8)
plt.plot(df.index, df['MA_20'], label='MA20', alpha=0.8)
plt.plot(df.index, df['MA_60'], label='MA60', alpha=0.8)

plt.title(f'{symbol} 股价与移动平均线')
plt.xlabel('日期')
plt.ylabel('价格 ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 计算金叉死叉信号
df['MA5_above_MA20'] = df['MA_5'] > df['MA_20']
df['Golden_Cross'] = (df['MA5_above_MA20'] == True) & (df['MA5_above_MA20'].shift(1) == False)
df['Death_Cross'] = (df['MA5_above_MA20'] == False) & (df['MA5_above_MA20'].shift(1) == True)

print(f"\n期间金叉次数: {df['Golden_Cross'].sum()}")
print(f"期间死叉次数: {df['Death_Cross'].sum()}")

## 5. 技术指标分析

计算RSI、MACD、布林带等常用技术指标

In [ ]:
# 计算技术指标
df = loader.calculate_technical_indicators(df)

# 可视化RSI
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# RSI
axes[0].plot(df.index, df['RSI'], label='RSI', color='purple')
axes[0].axhline(y=70, color='r', linestyle='--', label='超买线(70)')
axes[0].axhline(y=30, color='g', linestyle='--', label='超卖线(30)')
axes[0].fill_between(df.index, 30, 70, alpha=0.1, color='gray')
axes[0].set_title('RSI (相对强弱指标)')
axes[0].set_ylabel('RSI')
axes[0].set_ylim(0, 100)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MACD
axes[1].plot(df.index, df['MACD'], label='MACD', color='blue')
axes[1].plot(df.index, df['MACD_Signal'], label='Signal', color='red')
colors = ['green' if val >= 0 else 'red' for val in df['MACD_Histogram']]
axes[1].bar(df.index, df['MACD_Histogram'], label='Histogram', color=colors, alpha=0.5)
axes[1].set_title('MACD')
axes[1].set_ylabel('MACD')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 布林带
axes[2].plot(df.index, df['Close'], label='收盘价', color='black', alpha=0.8)
axes[2].plot(df.index, df['BB_Upper'], label='上轨', color='red', linestyle='--')
axes[2].plot(df.index, df['BB_Middle'], label='中轨', color='blue', linestyle='--')
axes[2].plot(df.index, df['BB_Lower'], label='下轨', color='green', linestyle='--')
axes[2].fill_between(df.index, df['BB_Lower'], df['BB_Upper'], alpha=0.1, color='gray')
axes[2].set_title('布林带')
axes[2].set_xlabel('日期')
axes[2].set_ylabel('价格 ($)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 风险指标计算

计算夏普比率、最大回撤、卡玛比率等风险调整收益指标

In [ ]:
# 计算风险指标
returns = df['Log_Return'].dropna()
prices = df['Close']

sharpe = PortfolioAnalyzer.calculate_sharpe_ratio(returns)
max_dd = PortfolioAnalyzer.calculate_max_drawdown(prices)
calmar = PortfolioAnalyzer.calculate_calmar_ratio(returns, prices)

print("===== 风险指标分析 =====")
print(f"夏普比率: {sharpe:.4f}")
print(f"  → 解释: 每承担一单位风险获得的超额收益")
print(f"  → 评估: >1优秀, >2卓越, >3非凡")
print()
print(f"最大回撤: {max_dd:.4f} ({max_dd*100:.2f}%)")
print(f"  → 解释: 从峰值到谷底的最大亏损")
print()
print(f"卡玛比率: {calmar:.4f}")
print(f"  → 解释: 年化收益与最大回撤的比值")
print(f"  → 评估: >2较好, >3优秀")

# 可视化回撤
peak = prices.cummax()
drawdown = (prices - peak) / peak

plt.figure(figsize=(14, 5))
plt.fill_between(df.index, drawdown, 0, color='red', alpha=0.3)
plt.plot(df.index, drawdown, color='darkred', linewidth=1)
plt.title('回撤曲线')
plt.xlabel('日期')
plt.ylabel('回撤幅度')
plt.grid(True, alpha=0.3)
plt.show()

## 7. 相关性分析

分析不同指标之间的相关性

In [ ]:
# 选择关键指标进行相关性分析
corr_cols = ['Close', 'Volume', 'Log_Return', 'Volatility_20d', 'RSI', 'MACD']
corr_matrix = df[corr_cols].corr()

# 绘制热力图
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('指标相关性热力图')
plt.tight_layout()
plt.show()

print("\n关键发现:")
print(f"- 价格与成交量的相关性: {corr_matrix.loc['Close', 'Volume']:.3f}")
print(f"- 收益率与波动率的相关性: {corr_matrix.loc['Log_Return', 'Volatility_20d']:.3f}")
print(f"- RSI与MACD的相关性: {corr_matrix.loc['RSI', 'MACD']:.3f}")

## 8. 保存处理后的数据

将处理好的数据保存，供后续建模使用

In [ ]:
# 保存数据
df.to_csv('../data/aapl_processed.csv')
print("数据已保存到 ../data/aapl_processed.csv")

# 显示最终数据信息
print(f"\n最终数据形状: {df.shape}")
print(f"\n数据列:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

---

## 学习总结

通过本Notebook，你应该掌握了：

1. **数据获取**：使用yfinance获取股票数据
2. **收益率计算**：理解简单收益率和对数收益率的区别和应用场景
3. **波动率分析**：掌握滚动波动率的计算方法及其金融意义
4. **技术指标**：了解RSI、MACD、布林带等常用指标的计算和解读
5. **风险评估**：学会计算夏普比率、最大回撤等风险指标

**下一步**：进入第二阶段，学习机器学习模型的构建和特征工程。